In [1]:
# import necessary libraries
import os
import torch 
import torch.nn as nn
from spnc import spnc_anisotropy
import numpy as np
import matplotlib.pyplot as plt
import tqdm as tqdm
import pickle
import spnc_ml as ml


from pathlib import Path


CANDIDATES = [
    
    Path(r"C:\Users\tom\Desktop\Repository"),
    Path(r"C:\Users\Chen\Desktop\Repository"),
    Path(r"/Users/vvvp./Desktop"),
]
searchpaths = [p for p in CANDIDATES if p.exists()]

#tuple of repos
repos = ('machine_learning_library',)

from deterministic_mask import fixed_seed_mask, max_sequences_mask
import repo_tools
repo_tools.repos_path_finder(searchpaths, repos)
from single_node_res import single_node_reservoir
import ridge_regression as RR
from linear_layer import *
from mask import binary_mask
from utility import *
from NARMA10 import NARMA10
from datasets.load_TI46_digits import *
import datasets.load_TI46 as TI46
from sklearn.metrics import classification_report

In [2]:
def gen_KR_GR_input(Nreadouts, Nwash=10, seed=1234):
    # set seed
    np.random.seed(seed)
    # generate KR inputs
    KR_inputs = np.random.ranf((Nreadouts, Nwash))
    GR_inputs = np.tile(np.random.ranf((10)), (Nreadouts,1))
    all_inputs = np.concatenate((KR_inputs, GR_inputs), axis=1)
    # 打印all_inputs的前10个元素
    return all_inputs


def Evaluate_KR_GR(states, Nreadouts, threshold=0.1):
    GR_states = states[:,-1,:]
    '''
    Change the last 7 columns to GR states, the rest are KR states
    '''
    KR_states = states[:,-11,:]
    uGR, sGR, vGR = np.linalg.svd(GR_states)
    uKR, sKR, vKR = np.linalg.svd(KR_states)
    KR = 0
    GR = 0
    for i in range(Nreadouts):
        if sGR[i]>threshold:
            GR += 1
        if sKR[i]>threshold:
            KR += 1
    return KR, GR

# ------------------------ Reservoir ----------------------------
def RunSpnc(signal,Nin,Nout,Nvirt,m0,transform, params,**kwargs):
    '''
    Run a reservoir computer with the signal sequence
    '''
    snr = single_node_reservoir(Nin, Nout, Nvirt, m0, res=transform)

    fixed_mask = kwargs.get('fixed_mask', False)
    if fixed_mask==True:
        # print("Deterministic mask will be used")
        seed_mask = kwargs.get('seed_mask', 1234)
        if seed_mask>=0:
            # print(seed_mask)
            snr.M = fixed_seed_mask(Nin, Nvirt, m0, seed=seed_mask)
        else:
            # print("Max_sequences mask will be used")
            snr.M = max_sequences_mask(Nin, Nvirt, m0)
    
    # Run
    S,_ = snr.transform(signal,params)
    
    return S

In [3]:
def evaluate_KRandGR(reservoir_params, Nreadouts=50, Nwash=10, **kwargs):
    
    Nreadouts= reservoir_params.Nvirt

    inputs = gen_KR_GR_input(Nreadouts, Nwash, seed=1234)   # <--- 用Nreadouts
    outputs = []
    for input_row in inputs:
        input_row = input_row.reshape(-1, 1)
        spn = spnc_anisotropy(reservoir_params.h, reservoir_params.theta_H,
                              reservoir_params.k_s_0, reservoir_params.phi,
                              reservoir_params.beta_prime, restart=True)
        transforms = spn.gen_signal_slow_delayed_feedback
        output = RunSpnc(input_row, 1, 1, reservoir_params.Nvirt,
                         reservoir_params.m0, transforms, reservoir_params.params, fixed_mask=True, seed_mask=1234)
        outputs.append(output)
    States = np.stack(outputs, axis=0)
    States = States/np.amax(States)
    KR, GR = Evaluate_KR_GR(States, Nreadouts, threshold=0.1)  # <--- 用Nreadouts
    return {'KR': KR, 'GR': GR}

In [4]:
# 构建储层对象
class ReservoirParams:
    def __init__(self, **kwargs):
            # Reservoir parameters 
            self.h = 0.4
            self.theta_H = 90
            self.k_s_0 = 0
            self.phi = 45
            self.beta_prime = 35.13826524755751

            # Network parameters 
            self.Nvirt = 50
            self.m0 = 0.005288612874870094
            self.bias = True
            self.Nwarmup = 0
            self.verbose_repr = False

            self.params = {
                'theta': 0.34142235979698393,
                'gamma': 0.069274461903986,
                'delay_feedback': 0,
                'Nvirt': self.Nvirt,
                'length_warmup': self.Nwarmup,
                'warmup_sample': self.Nwarmup * self.Nvirt,
                'voltage_noise': False,
                'seed_voltage_noise': 1234,
                'delta_V': 0.1,
                'johnson_noise': False,
                'seed_johnson_noise': 1234,
                'mean_johnson_noise': 0.0000,
                'std_johnson_noise': 0.00001,
                'thermal_noise': False,
                'seed_thermal_noise': 1234,
                'lambda_ou': 1.0,
                'sigma_ou': 0.1
        }

            for key in ['h', 'theta_H', 'k_s_0', 'phi', 'beta_prime', 'Nvirt', 'm0', 'bias', 'Nwarmup']:
                if key in kwargs:
                    setattr(self, key, kwargs[key])

            
            if 'params' in kwargs and isinstance(kwargs['params'], dict):
                self.params.update(kwargs['params'])

    
    def update_params(self, **kwargs):
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            if key in self.params:
                self.params[key] = value
            if not hasattr(self, key) and key not in self.params:
                raise AttributeError(f"ReservoirParams has no attribute or param key '{key}'")
            
    def print_params(self, verbose=False):
        if not verbose:
            print(f"ReservoirParams(h={self.h}, beta_prime={self.beta_prime}, Nvirt={self.Nvirt})")
        else:
            print(f"ReservoirParams detailed info:")
            print(f"  h = {self.h}")
            print(f"  theta_H = {self.theta_H}")
            print(f"  k_s_0 = {self.k_s_0}")
            print(f"  phi = {self.phi}")
            print(f"  beta_prime = {self.beta_prime}")
            print(f"  Nvirt = {self.Nvirt}")
            print(f"  m0 = {self.m0}")
            print(f"  bias = {self.bias}")
            print("  params dictionary:")
            for k, v in self.params.items():
                print(f"    {k}: {v}")

In [5]:
params_bestCQ = ReservoirParams(
        h=0.4, m0=0.006937322149792008, Nvirt=200, beta_prime=27.251620432439488,
        params={'theta': 0.01, 'gamma': 0.3663969812988086, 'Nvirt': 200}
    )

In [ ]:
results = evaluate_KRandGR(params_bestCQ)


In [9]:
print(results)

{'KR': 140, 'GR': 1}


In [5]:
params_bestMC = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.1564938388583194, 'gamma': 0.04608425844940916, 'Nvirt': 200}
    )

In [6]:
results_bestMC = evaluate_KRandGR(params_bestMC)
print(results_bestMC)

{'KR': 6, 'GR': 3}


In [7]:
params_bestPhase = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.07883177553412853, 'gamma': 0.09737503590304286, 'Nvirt': 200}
    )

In [8]:
results_bestPhase = evaluate_KRandGR(params_bestPhase)
print(results_bestPhase)

{'KR': 6, 'GR': 1}


In [9]:
params_109 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	39.70001479261286,
        params={'theta': 0.14799158258968137, 'gamma': 0.05936342628024845, 'Nvirt': 200}
    )

In [10]:
results_109 = evaluate_KRandGR(params_109)
print(results_109)

{'KR': 6, 'GR': 3}


In [11]:
params_113 = ReservoirParams(
        h=0.4, m0=0.001, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.03442997633689481, 'gamma': 0.09508576109692872, 'Nvirt': 200})
results_113 = evaluate_KRandGR(params_113)
print(results_113)

{'KR': 45, 'GR': 1}


In [12]:
params_29 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.11577824609314408, 'gamma': 0.04447510407651761, 'Nvirt': 200})
results_29 = evaluate_KRandGR(params_29)
print(results_29)

{'KR': 6, 'GR': 3}


In [13]:
params_214 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.20890131642287738, 'gamma': 0.14586775378011477, 'Nvirt': 200}
    )
results_214 = evaluate_KRandGR(params_214)
print(results_214)

{'KR': 6, 'GR': 5}


In [14]:
params_138 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.6, 'gamma': 0.1172260326884624, 'Nvirt': 200}
    )
results_138 = evaluate_KRandGR(params_138)
print(results_138)

{'KR': 5, 'GR': 3}


In [15]:
params_326 = ReservoirParams(
        h=0.4, m0=0.07187768090200536, Nvirt=200, beta_prime= 28.21081994351713,
        params={'theta': 0.10419727159331738, 'gamma': 0.07187768090200536, 'Nvirt': 200}
    )
results_326 = evaluate_KRandGR(params_326)
print(results_326)

{'KR': 7, 'GR': 1}
